In [ ]:
# %pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift
import time
notebook_start = time.perf_counter()

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import gpytorch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score as _r2, mean_squared_error as _mse
import thermoift.PLOT_SETTINGS as ps
from thermoift import MLPostprocessing, MLPreprocessing
from thermoift.rng_utils import get_rng
from thermoift import plot_correlation_heatmap, print_model_metrics

print(f"PyTorch  : {torch.__version__}")
print(f"GPyTorch : {gpytorch.__version__}")

In [ ]:
OUTPUT_FOLDER    = "SVGP_RESIDUAL_OUTPUTS"
SEED             = 4555525

# SVGP hyperparameters
N_INDUCING       = 1500   # total inducing points
N_INDUCING_TAIL  = 0.30   # fraction from high-|residual| tail
TAIL_THRESHOLD   = 0.5    # |residual| threshold (mN/m) for tail inducing pts + oversampling
TAIL_OVERSAMPLE  = 5      # mini-batch weight for tail samples (was 3)
N_EPOCHS         = 200    # training epochs
BATCH_SIZE       = 1024   # mini-batch size
LR               = 0.01   # initial Adam LR (cosine-annealed to LR_MIN)
LR_MIN           = 1e-4   # final LR

# Cross-validation settings
RUN_CV    = False
CV_FOLDS  = 5
CV_EPOCHS = 30


In [ ]:
df         = pd.read_csv("1691761_CombinedDataset_A3.csv")
df.columns = [col.strip().replace(" ", "_") for col in df.columns]

# Mixture CO2 partial density gap
df["NUM1"] = df["rhoL_carbon_dioxide"] - df["rhoV_carbon_dioxide"]
df["DEN1"] = df["rhoL0_carbon_dioxide"] - df["rhoV0_carbon_dioxide"]

# Reference density-gap ratio
df["r1"]   = df["NUM1"] / df["DEN1"]
df["r1sq"] = df["r1"] ** 2

# WSD baseline
df["gamma_base"] = df["gamma0_carbon_dioxide"] * df["r1sq"]
df["gamma_cDFT"] = df["gamma_wsd"] + df["gamma_cDFT_minus_wsd_uncorrected"]

# Target: relative residual vs baseline
df             = df.dropna(subset=["gamma_cDFT", "gamma_base"]).copy()
df["residual"] = df["gamma_cDFT"] - df["gamma_base"]

print(f"Total samples (full dataset — no cap): {len(df)}")
print(f"\nResidual statistics:")
print(df["residual"].describe())

In [ ]:
target   = "residual"
rng      = get_rng(seed=SEED)

z_columns     = [col for col in df.columns if col.startswith("z_")]
DROP_FEATURES = ["z_oxygen"]
Z_non_zero    = [col for col in z_columns if (df[col] != 0).any() and col not in DROP_FEATURES]
features      = ["T", "P"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

In [ ]:
plot_correlation_heatmap(df, features, target, save_path="SVGP_residual_correlation", folder=OUTPUT_FOLDER)

In [ ]:
prep = MLPreprocessing(df=df, features=features, target=target)

fig_T, ax_T = prep.plot_scatter(x="T", y=target, color_by="P")
ax_T.axhline(0.0, color="black", linestyle=":", linewidth=1.0)
ps.save_plot(fig_T, "SVGP_residual_vs_T", folder=OUTPUT_FOLDER)

fig_P, ax_P = prep.plot_scatter(x="P", y=target, color_by="T")
ax_P.axhline(0.0, color="black", linestyle=":", linewidth=1.0)
ps.save_plot(fig_P, "SVGP_residual_vs_P", folder=OUTPUT_FOLDER)

In [ ]:
print(df[["gamma_cDFT", "gamma_base", "residual"]].isna().sum())
print("Any NaN in X:", X.isna().any().any())
print("Any NaN in y:", y.isna().any())

In [ ]:
from sklearn.preprocessing import QuantileTransformer

# ── QuantileTransformer target transform ─────────────────────────────────────
# Maps the empirical CDF of y_train to N(0,1): every quantile contributes
# equally to the ELBO, so the sparse positive tail is no longer outweighed
# by the dense negative bulk.  Fit on train only; transform test/val.
qt = QuantileTransformer(
    output_distribution="normal",
    n_quantiles=min(len(y_train), 1000),
    random_state=SEED,
)
y_train_qt = qt.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_qt  = qt.transform(y_test.values.reshape(-1, 1)).ravel()
y_val_qt   = qt.transform(y_val.values.reshape(-1, 1)).ravel()

# StandardScaler on top (keeps pipeline consistent; ≈ no-op since QT → N(0,1))
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)
X_val_scaled   = scaler.transform(X_val).astype(np.float32)
X_all_scaled   = scaler.transform(X).astype(np.float32)

y_mean_val = float(y_train_qt.mean())
y_std_val  = float(y_train_qt.std())

y_train_norm = ((y_train_qt - y_mean_val) / y_std_val).astype(np.float32)
y_test_norm  = ((y_test_qt  - y_mean_val) / y_std_val).astype(np.float32)
y_val_norm   = ((y_val_qt   - y_mean_val) / y_std_val).astype(np.float32)

# Tensors
X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor  = torch.from_numpy(X_test_scaled)
X_val_tensor   = torch.from_numpy(X_val_scaled)
y_train_tensor = torch.from_numpy(y_train_norm)

print(f"X_train tensor: {X_train_tensor.shape}  dtype: {X_train_tensor.dtype}")
print(f"y QT normalisation — mean: {y_mean_val:.4f}  std: {y_std_val:.4f}")


In [ ]:
class SVGPModel(gpytorch.models.ApproximateGP):
    """Stochastic Variational GP with ARD Matern-5/2 kernel."""

    def __init__(self, inducing_points: torch.Tensor):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)
        self.mean_module  = gpytorch.means.ConstantMean()
        # Matern 5/2 with ARD — heavier tails than RBF, better extrapolation into
        # the sparse high-residual regime
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(nu=2.5, ard_num_dims=inducing_points.shape[1])
        )

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x  = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


def init_inducing_points(X_scaled: np.ndarray, n_inducing: int, seed: int,
                         y_raw: np.ndarray | None = None,
                         tail_frac: float = 0.20,
                         tail_threshold: float = 1.0) -> torch.Tensor:
    """Stratified inducing point initialisation.

    80 % from MiniBatchKMeans on the full training set (dense-region coverage).
    20 % randomly sampled from the high-|residual| tail (|y| > tail_threshold)
    to ensure the model has anchors in the extreme-value regime.
    """
    rng = np.random.default_rng(seed)
    n_tail  = max(1, int(round(n_inducing * tail_frac)))
    n_bulk  = n_inducing - n_tail

    # Bulk: k-means centroids
    km = MiniBatchKMeans(n_clusters=n_bulk, random_state=seed, n_init=3, batch_size=4096)
    km.fit(X_scaled)
    bulk_pts = km.cluster_centers_  # (n_bulk, d)

    # Tail: random draw from high-|residual| samples
    if y_raw is not None:
        tail_mask = np.abs(y_raw) > tail_threshold
        n_tail_avail = tail_mask.sum()
        if n_tail_avail >= n_tail:
            tail_idx = rng.choice(np.where(tail_mask)[0], size=n_tail, replace=False)
        else:
            tail_idx = rng.choice(len(X_scaled), size=n_tail, replace=False)
            print(f"  [warn] only {n_tail_avail} tail samples; falling back to random draw")
        tail_pts = X_scaled[tail_idx]
    else:
        tail_idx = rng.choice(len(X_scaled), size=n_tail, replace=False)
        tail_pts = X_scaled[tail_idx]

    all_pts = np.vstack([bulk_pts, tail_pts]).astype(np.float32)
    print(f"  Inducing pts: {n_bulk} bulk (k-means) + {n_tail} tail = {len(all_pts)} total")
    return torch.tensor(all_pts, dtype=torch.float32)


def predict_batched(model, likelihood, X_tensor: torch.Tensor,
                    batch_size: int = 2048) -> tuple[np.ndarray, np.ndarray]:
    """Batched prediction to avoid OOM on large datasets."""
    model.eval()
    likelihood.eval()
    means, stds = [], []
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        for i in range(0, len(X_tensor), batch_size):
            xb   = X_tensor[i : i + batch_size]
            pred = likelihood(model(xb))
            means.append(pred.mean.numpy())
            stds.append(pred.stddev.numpy())
    return np.concatenate(means), np.concatenate(stds)


print("SVGPModel (Matern-5/2 ARD), stratified init_inducing_points, and predict_batched defined.")


In [ ]:
# Stratified inducing points: 70 % k-means bulk + 30 % tail (|residual| > TAIL_THRESHOLD)
inducing_points = init_inducing_points(
    X_train_scaled, N_INDUCING, SEED,
    y_raw=y_train.values, tail_frac=N_INDUCING_TAIL, tail_threshold=TAIL_THRESHOLD,
)
print(f"Inducing points shape: {inducing_points.shape}")

# Build model and likelihood
torch.manual_seed(SEED)
svgp_model = SVGPModel(inducing_points)
likelihood = gpytorch.likelihoods.GaussianLikelihood()

# ELBO objective
mll = gpytorch.mlls.VariationalELBO(likelihood, svgp_model, num_data=len(X_train_tensor))

# Optimizer — joint over model + likelihood parameters
optimizer = torch.optim.Adam(
    list(svgp_model.parameters()) + list(likelihood.parameters()),
    lr=LR,
)

# Cosine annealing: smoothly decays LR from LR to LR_MIN over N_EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=LR_MIN)

# ── Weighted DataLoader — oversample tail by TAIL_OVERSAMPLE ─────────────────
# Samples with |residual| > TAIL_THRESHOLD get weight TAIL_OVERSAMPLE; bulk gets 1.
# This ensures tail contributes ~25 % of each mini-batch instead of ~5 %.
y_train_abs = np.abs(y_train.values)
sample_weights = np.where(y_train_abs > TAIL_THRESHOLD,
                          float(TAIL_OVERSAMPLE), 1.0).astype(np.float32)
sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)

n_tail_samples = int((y_train_abs > TAIL_THRESHOLD).sum())
eff_tail_pct   = 100 * TAIL_OVERSAMPLE * n_tail_samples / (
    TAIL_OVERSAMPLE * n_tail_samples + (len(y_train_abs) - n_tail_samples))
print(f"Tail samples (|y|>{TAIL_THRESHOLD}): {n_tail_samples} / {len(y_train_abs)} "
      f"→ effective mini-batch share ≈ {eff_tail_pct:.1f}%")

total_params = sum(p.numel() for p in svgp_model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Batches per epoch: {len(train_loader)}")


In [ ]:
svgp_model.train()
likelihood.train()

losses = []
train_start = time.perf_counter()

for epoch in range(1, N_EPOCHS + 1):
    epoch_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = svgp_model(x_batch)
        loss   = -mll(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()  # cosine LR decay
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    if epoch % 10 == 0 or epoch == 1:
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d}/{N_EPOCHS} | ELBO loss: {avg_loss:.4f} | LR: {current_lr:.2e}")

train_elapsed = time.perf_counter() - train_start
print(f"\nTraining time: {train_elapsed/60:.2f} min")


In [ ]:
# ELBO loss curve
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, N_EPOCHS + 1), losses, color="steelblue", linewidth=1.2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Negative ELBO loss")
ax.set_title("SVGP Training Loss")
ax.xaxis.set_minor_locator(plt.MultipleLocator(5))
plt.tight_layout()
ps.save_plot(fig, "SVGP_training_loss", folder=OUTPUT_FOLDER)
plt.show()

In [ ]:
# Batched predictions — undo StandardScaler then invert QuantileTransformer
y_train_pred_norm, y_train_std_norm = predict_batched(svgp_model, likelihood, X_train_tensor)
y_test_pred_norm,  y_test_std_norm  = predict_batched(svgp_model, likelihood, X_test_tensor)
y_val_pred_norm,   y_val_std_norm   = predict_batched(svgp_model, likelihood, X_val_tensor)

# Step 1: undo StandardScaler (back to QT-transformed space)
y_train_pred_qt = y_train_pred_norm * y_std_val + y_mean_val
y_test_pred_qt  = y_test_pred_norm  * y_std_val + y_mean_val
y_val_pred_qt   = y_val_pred_norm   * y_std_val + y_mean_val

# Step 2: invert QuantileTransformer → original mN/m units
def qt_inverse(arr):
    return qt.inverse_transform(arr.reshape(-1, 1)).ravel()

y_train_pred = qt_inverse(y_train_pred_qt)
y_test_pred  = qt_inverse(y_test_pred_qt)
y_val_pred   = qt_inverse(y_val_pred_qt)

# Step 3: propagate std via numerical Jacobian of QT inverse
# dy_orig/dy_qt ≈ [QT^{-1}(y+ε) - QT^{-1}(y-ε)] / (2ε)
eps = 1e-4
def qt_jacobian(arr):
    return np.abs(qt_inverse(arr + eps) - qt_inverse(arr - eps)) / (2 * eps)

y_train_std = y_train_std_norm * y_std_val * qt_jacobian(y_train_pred_qt)
y_test_std  = y_test_std_norm  * y_std_val * qt_jacobian(y_test_pred_qt)
y_val_std   = y_val_std_norm   * y_std_val * qt_jacobian(y_val_pred_qt)

# Metrics
metrics = print_model_metrics(
    y_train, y_train_pred,
    y_test,  y_test_pred,
    target,  unit="mN/m",
    y_val=y_val, y_val_pred=y_val_pred,
)


In [ ]:
# ARD feature importances: inverse length-scale (same interpretation as GPR)
_length_scales       = svgp_model.covar_module.base_kernel.lengthscale.squeeze().detach().numpy()
_feature_importances = 1.0 / _length_scales
_feature_importances = _feature_importances / _feature_importances.sum()

post = MLPostprocessing(
    y_true=y_test,
    y_pred=y_test_pred,
    target="delta_gamma",
    feature_importances=_feature_importances,
    feature_names=features,
    datasets={
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)

In [ ]:
post.plot_parity(model_name="SVGP (RBF-ARD)", save_path="SVGP_gamma_parity_plot", folder=OUTPUT_FOLDER)

In [ ]:
post.plot_residual_distribution(save_path="SVGP_gamma_residual_distribution", folder=OUTPUT_FOLDER)

In [ ]:
post.plot_residual_vs_predicted(save_path="SVGP_gamma_residual_vs_predicted", folder=OUTPUT_FOLDER)

In [ ]:
post.plot_std_histogram(y_test_std, save_path="SVGP_gamma_std_histogram", folder=OUTPUT_FOLDER)

In [ ]:
post.plot_parity_colored_by_std(y_test_std, save_path="SVGP_gamma_parity_colored_std", folder=OUTPUT_FOLDER)

In [ ]:
post.plot_error_vs_std(y_test_std, save_path="SVGP_gamma_error_vs_std", folder=OUTPUT_FOLDER)

In [ ]:
# Response curves — wrap the SVGP in a predict-only callable compatible with MLPostprocessing
class SVGPWrapper:
    """Thin sklearn-style wrapper so MLPostprocessing.plot_response_curves works unchanged."""
    def __init__(self, model, likelihood, scaler, y_mean, y_std):
        self._model      = model
        self._likelihood = likelihood
        self._scaler     = scaler
        self._y_mean     = y_mean
        self._y_std      = y_std

    def predict(self, X, return_std=False):
        if not isinstance(X, np.ndarray):
            X = np.asarray(X, dtype=np.float32)
        Xs  = self._scaler.transform(X).astype(np.float32)
        Xt  = torch.from_numpy(Xs)
        m, s = predict_batched(self._model, self._likelihood, Xt)
        m_orig = m * self._y_std + self._y_mean
        s_orig = s * self._y_std
        return (m_orig, s_orig) if return_std else m_orig


svgp_wrapper = SVGPWrapper(svgp_model, likelihood, scaler, y_mean_val, y_std_val)

X_ref = X_train.median().values
post.plot_response_curves(
    model               = svgp_wrapper,
    X_ref               = X_ref,
    feature_names       = features,
    X_train             = X_train,
    n_points            = 100,
    return_std          = True,
    save_individually   = True,
    save_path           = "SVGP_gamma_response",
    folder              = OUTPUT_FOLDER,
)

In [ ]:
# 2D Response Surface: T vs P (all z_* fixed at training median)
post.plot_2d_response_surface(
    model         = svgp_wrapper,
    X_train       = X_train,
    y_train       = y_train,
    feature_names = features,
    T_name        = "T",
    P_name        = "P",
    n_grid        = 80,
    return_std    = False,
    save_path     = "SVGP_2D_response_T_P",
    folder        = OUTPUT_FOLDER,
)


In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

model_state = {
    "model_state_dict":      svgp_model.state_dict(),
    "likelihood_state_dict": likelihood.state_dict(),
    "scaler":                scaler,
    "qt":                    qt,           # QuantileTransformer (replaces signlog)
    "y_mean":                y_mean_val,
    "y_std":                 y_std_val,
    "y_transform":           "quantile_normal",
    "n_inducing":            N_INDUCING,
    "features":              features,
    "target":                target,
}
model_path = os.path.join(OUTPUT_FOLDER, "SVGP_residual_model.pt")
torch.save(model_state, model_path)
print(f"Model saved to: {model_path}")


In [ ]:
# Reconstruct full gamma_cDFT = gamma_base + residual_pred
gamma_base_train = df.loc[X_train.index, "gamma_base"].values
gamma_base_test  = df.loc[X_test.index,  "gamma_base"].values
gamma_base_val   = df.loc[X_val.index,   "gamma_base"].values

gamma_cDFT_train = df.loc[X_train.index, "gamma_cDFT"].values
gamma_cDFT_test  = df.loc[X_test.index,  "gamma_cDFT"].values
gamma_cDFT_val   = df.loc[X_val.index,   "gamma_cDFT"].values

gamma_pred_train = gamma_base_train + y_train_pred
gamma_pred_test  = gamma_base_test  + y_test_pred
gamma_pred_val   = gamma_base_val   + y_val_pred

print(f"Reconstructed gamma_cDFT  R² — train: {_r2(gamma_cDFT_train, gamma_pred_train):.4f} "
      f"| test: {_r2(gamma_cDFT_test, gamma_pred_test):.4f} "
      f"| val:  {_r2(gamma_cDFT_val,  gamma_pred_val):.4f}")

In [ ]:
post.plot_reconstructed_parity(
    datasets_reconstructed={
        "train": (gamma_cDFT_train, gamma_pred_train, y_train_std),
        "test":  (gamma_cDFT_test,  gamma_pred_test,  y_test_std),
        "val":   (gamma_cDFT_val,   gamma_pred_val,   y_val_std),
    },
    n_sigma=2.0,
    model_name="SVGP",
    save_path="SVGP_gamma_reconstructed_parity",
    folder=OUTPUT_FOLDER,
)

In [ ]:
# 5-fold cross-validation (manual — sklearn cross_val_score is not compatible with GPyTorch)
# Uses CV_EPOCHS (reduced) for feasibility; set RUN_CV=False to skip.
if RUN_CV:
    kf             = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    cv_r2_scores   = np.zeros(CV_FOLDS)
    cv_rmse_scores = np.zeros(CV_FOLDS)

    X_all_np = X_all_scaled                         # already scaled, float32
    y_all_np = y.values.astype(np.float32)           # original units

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all_np)):
        Xf_tr, yf_tr = X_all_np[tr_idx], y_all_np[tr_idx]
        Xf_va, yf_va = X_all_np[va_idx], y_all_np[va_idx]

        # Per-fold y normalisation
        yf_mean = float(yf_tr.mean())
        yf_std  = float(yf_tr.std())
        yf_tr_norm = torch.tensor((yf_tr - yf_mean) / yf_std, dtype=torch.float32)

        Xf_tr_t = torch.from_numpy(Xf_tr)
        Xf_va_t = torch.from_numpy(Xf_va)

        # Inducing points from fold training data
        ip_fold = init_inducing_points(Xf_tr, N_INDUCING, SEED + fold)

        torch.manual_seed(SEED + fold)
        fold_model      = SVGPModel(ip_fold)
        fold_likelihood = gpytorch.likelihoods.GaussianLikelihood()
        fold_mll        = gpytorch.mlls.VariationalELBO(
                              fold_likelihood, fold_model, num_data=len(Xf_tr_t))
        fold_optimizer  = torch.optim.Adam(
                              list(fold_model.parameters()) + list(fold_likelihood.parameters()),
                              lr=LR)

        fold_loader = DataLoader(
            TensorDataset(Xf_tr_t, yf_tr_norm),
            batch_size=BATCH_SIZE, shuffle=True,
        )

        fold_model.train()
        fold_likelihood.train()
        for _ in range(CV_EPOCHS):
            for xb, yb in fold_loader:
                fold_optimizer.zero_grad()
                (-fold_mll(fold_model(xb), yb)).backward()
                fold_optimizer.step()

        # Predict and un-normalise
        yf_pred_norm, _ = predict_batched(fold_model, fold_likelihood, Xf_va_t)
        yf_pred          = yf_pred_norm * yf_std + yf_mean

        cv_r2_scores[fold]   = _r2(yf_va, yf_pred)
        cv_rmse_scores[fold] = float(np.sqrt(_mse(yf_va, yf_pred)))
        print(f"  Fold {fold+1}/{CV_FOLDS} | R²: {cv_r2_scores[fold]:.6f} | RMSE: {cv_rmse_scores[fold]:.6f}")

    print(f"\nCross-Validation R² Scores:   {cv_r2_scores}")
    print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
    print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
    print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
else:
    cv_r2_scores   = np.array([])
    cv_rmse_scores = np.array([])
    print("CV skipped (RUN_CV=False).")

In [ ]:
post.print_summary()

# Collect and save all metrics
metrics["cv_r2_scores"]         = cv_r2_scores.tolist()
metrics["cv_r2_mean"]           = float(cv_r2_scores.mean()) if len(cv_r2_scores) else None
metrics["cv_r2_std"]            = float(cv_r2_scores.std())  if len(cv_r2_scores) else None
metrics["cv_rmse_scores"]       = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]         = float(cv_rmse_scores.mean()) if len(cv_rmse_scores) else None
metrics["cv_rmse_std"]          = float(cv_rmse_scores.std())  if len(cv_rmse_scores) else None
metrics["model"]                = "SVGP_RBF_ARD"
metrics["kernel"]               = str(svgp_model.covar_module)
metrics["n_inducing"]           = N_INDUCING
metrics["n_epochs"]             = N_EPOCHS
metrics["batch_size"]           = BATCH_SIZE
metrics["lr"]                   = LR
metrics["features"]             = features
metrics["target"]               = target
metrics["seed"]                 = SEED

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
metrics_path = os.path.join(OUTPUT_FOLDER, f"SVGP_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print(f"\nMetrics saved to: {metrics_path}")

In [ ]:
notebook_end     = time.perf_counter()
elapsed_minutes  = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")